# 01 — Clean Historical Data

**Goal:** Load all historical indicator files, clean them, merge into one panel dataset.  
**Reads:** All CSV/XLSX files in `data/raw/`  
**Outputs:** `data/processed/historical_panel.csv`  

Variables collected:
- GDP (current US$) → used to compute GDP per capita
- Population (from SSP Excel, Historical Reference scenario)
- HDI (Human Development Index)
- Control of Corruption
- Employment in Agriculture (% of total employment)
- Gini coefficient
- Poverty headcount ratios at $3, $8.30, $10/day thresholds

> **Note:** The $4.20/day data has been temporarily excluded — the source file contains poverty *gap* data, not headcount ratios. Will be re-added once corrected data is sourced.

## Imports and paths

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
OUTPUTS_DIR = Path("../outputs")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

Directories ready.


## Helper function: load World Bank CSV

World Bank CSVs have 4 metadata rows, then a header row with:
`Country Name, Country Code, Indicator Name, Indicator Code, 1960, 1961, ..., 2024`

We melt from wide to long and filter out aggregate regions (e.g., "World", "Sub-Saharan Africa").

In [25]:
# World Bank aggregate codes to exclude (not real countries)
WB_AGGREGATE_CODES = {
    "AFE", "AFW", "ARB", "CEB", "CSS", "EAP", "EAR", "EAS", "ECA",
    "ECS", "EMU", "EUU", "FCS", "HIC", "HPC", "IBD", "IBT", "IDA",
    "IDB", "IDX", "INX", "LAC", "LCN", "LDC", "LIC", "LMC", "LMY",
    "LTE", "MEA", "MIC", "MNA", "NAC", "OED", "OSS", "PRE", "PSS",
    "PST", "SAS", "SSA", "SSF", "SST", "TEA", "TEC", "TLA", "TMN",
    "TSA", "TSS", "UMC", "WLD"
}

def load_wb_csv(filepath, var_name, year_start=1996, year_end=2023):
    """
    Load a World Bank CSV file and convert from wide to long format.
    Returns DataFrame with columns: country_name, country_code, year, <var_name>
    """
    df = pd.read_csv(filepath, skiprows=4)
    
    # Print indicator name for verification
    if "Indicator Name" in df.columns:
        print(f"  Indicator: {df['Indicator Name'].iloc[0]}")
    
    # Select year columns that exist in the file
    year_cols = [str(y) for y in range(year_start, year_end + 1) if str(y) in df.columns]
    keep_cols = ["Country Name", "Country Code"] + year_cols
    df = df[keep_cols]
    
    # Melt to long format
    df = df.melt(
        id_vars=["Country Name", "Country Code"],
        var_name="year",
        value_name=var_name
    )
    
    # Clean types
    df.rename(columns={"Country Name": "country_name", "Country Code": "country_code"}, inplace=True)
    df["year"] = df["year"].astype(int)
    df[var_name] = pd.to_numeric(df[var_name], errors="coerce")
    
    # Remove aggregate regions
    df = df[~df["country_code"].isin(WB_AGGREGATE_CODES)].reset_index(drop=True)
    
    print(f"  Shape: {df.shape}, Countries: {df['country_code'].nunique()}, "
          f"Non-null {var_name}: {df[var_name].notna().sum()}")
    return df

## Load GDP

In [26]:
print("Loading GDP...")
gdp = load_wb_csv(DATA_RAW / "GDP_HistoricalData_1960_2024.csv", "gdp")
gdp.head()

Loading GDP...
  Indicator: GDP (current US$)
  Shape: (6076, 4), Countries: 217, Non-null gdp: 5845


,country_name,country_code,year,gdp
0,Aruba,ABW,1996,1.379888e+09
1,Afghanistan,AFG,1996,NaN
2,Angola,AGO,1996,7.526422e+09
3,Albania,ALB,1996,3.234486e+09
4,Andorra,AND,1996,1.224024e+09


## Load Control of Corruption

In [27]:
print("Loading Control of Corruption...")
corruption = load_wb_csv(
    DATA_RAW / "ControlOfCorruption_Historical_WorldBank_1996-2023.csv",
    "control_of_corruption"
)
corruption.head()

Loading Control of Corruption...
  Indicator: Control of Corruption: Estimate
  Shape: (6076, 4), Countries: 217, Non-null control_of_corruption: 4988


,country_name,country_code,year,control_of_corruption
0,Aruba,ABW,1996,NaN
1,Afghanistan,AFG,1996,-1.291705
2,Angola,AGO,1996,-1.167702
3,Albania,ALB,1996,-0.893903
4,Andorra,AND,1996,1.318143


## Load Employment in Agriculture

In [28]:
print("Loading Employment in Agriculture...")
agriculture = load_wb_csv(
    DATA_RAW / "EmploymentInAgriculture_Historical_WorldBank_1991-2023.csv",
    "employment_agriculture"
)
agriculture.head()

Loading Employment in Agriculture...
  Indicator: Employment in agriculture (% of total employment) (modeled ILO estimate)
  Shape: (6076, 4), Countries: 217, Non-null employment_agriculture: 5232


,country_name,country_code,year,employment_agriculture
0,Aruba,ABW,1996,NaN
1,Afghanistan,AFG,1996,67.170502
2,Angola,AGO,1996,38.959080
3,Albania,ALB,1996,52.857857
4,Andorra,AND,1996,NaN


## Load Gini Coefficient

In [29]:
print("Loading Gini Coefficient...")
gini = load_wb_csv(
    DATA_RAW / "GiniCoefficient_Historical_WorldBank_1963-2024.csv",
    "gini"
)
gini.head()

Loading Gini Coefficient...
  Indicator: Gini index
  Shape: (6076, 4), Countries: 217, Non-null gini: 1999


,country_name,country_code,year,gini
0,Aruba,ABW,1996,NaN
1,Afghanistan,AFG,1996,NaN
2,Angola,AGO,1996,NaN
3,Albania,ALB,1996,27.0
4,Andorra,AND,1996,NaN


## Load Poverty Data

We have three poverty threshold files in **different formats**:
- `poverty_3$_1963_2024.csv` — World Bank format ($3.00/day headcount ratio)
- `POV_8.30$_1963_2024.csv` — Simple CSV (country, year, headcount_ratio_830)
- `POV_10$_1963_2024.csv` — Simple CSV (country, year, headcount_ratio_1000)

> **Excluded:** `pov_420_1963_2-24.csv` — contains poverty *gap* (not headcount ratio). Will be re-added once corrected.

In [30]:
# Poverty $3/day — World Bank format
print("Loading Poverty $3/day...")
pov3 = load_wb_csv(DATA_RAW / "poverty_3$_1963_2024.csv", "poverty_3")
print()

# Poverty $4.20/day — EXCLUDED: file contains poverty gap, not headcount ratio
# print("Loading Poverty $4.20/day...")
# pov420 = load_wb_csv(DATA_RAW / "pov_420_1963_2-24.csv", "poverty_4_20")
# print()

Loading Poverty $3/day...
  Indicator: Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)
  Shape: (6076, 4), Countries: 217, Non-null poverty_3: 1999



### Load poverty $8.30 and $10 (simple CSV format)

These files have columns `country, year, headcount_ratio_xxx` — no country codes.  
We build a name→code mapping from the GDP file (which has both).

In [31]:
# Build country name → code mapping from GDP data
name_to_code = (
    gdp[["country_name", "country_code"]]
    .drop_duplicates()
    .set_index("country_name")["country_code"]
    .to_dict()
)
print(f"Name-to-code mapping: {len(name_to_code)} entries")

Name-to-code mapping: 217 entries


In [32]:
def load_simple_poverty_csv(filepath, value_col_in_file, var_name):
    """
    Load a simple poverty CSV (country, year, value).
    Maps country names to codes using the GDP-derived mapping.
    """
    df = pd.read_csv(filepath)
    print(f"  Columns: {list(df.columns)}")
    
    df = df.rename(columns={"country": "country_name", value_col_in_file: var_name})
    df["year"] = df["year"].astype(int)
    df[var_name] = pd.to_numeric(df[var_name], errors="coerce")
    
    # Map country names to codes
    df["country_code"] = df["country_name"].map(name_to_code)
    
    # Report unmatched countries
    unmatched = df[df["country_code"].isna()]["country_name"].unique()
    if len(unmatched) > 0:
        print(f"  WARNING: {len(unmatched)} countries could not be matched: {unmatched[:10]}")
    
    # Keep only matched, filter to 1996-2023
    df = df.dropna(subset=["country_code"])
    df = df[(df["year"] >= 1996) & (df["year"] <= 2023)]
    df = df[["country_name", "country_code", "year", var_name]].reset_index(drop=True)
    
    print(f"  Shape: {df.shape}, Countries: {df['country_code'].nunique()}, "
          f"Non-null {var_name}: {df[var_name].notna().sum()}")
    return df

print("Loading Poverty $8.30/day...")
pov830 = load_simple_poverty_csv(
    DATA_RAW / "POV_8.30$_1963_2024.csv", "headcount_ratio_830", "poverty_8_30"
)
print()

print("Loading Poverty $10/day...")
pov10 = load_simple_poverty_csv(
    DATA_RAW / "POV_10$_1963_2024.csv", "headcount_ratio_1000", "poverty_10"
)

Loading Poverty $8.30/day...
  Columns: ['country', 'year', 'headcount_ratio_830']
 'China (urban)' 'Colombia (urban)' 'Congo' 'Democratic Republic of Congo'
 'East Asia and Pacific (WB)' 'East Timor']
  Shape: (1676, 4), Countries: 149, Non-null poverty_8_30: 1676

Loading Poverty $10/day...
  Columns: ['country', 'year', 'headcount_ratio_1000']
 'China (urban)' 'Colombia (urban)' 'Congo' 'Democratic Republic of Congo'
 'East Asia and Pacific (WB)' 'East Timor']
  Shape: (1676, 4), Countries: 149, Non-null poverty_10: 1676


## Load HDI (Human Development Index)

The HDI file is a simple CSV with `HDI Rank, Country, 1990, 1991, ..., 2019`.  
Missing values are coded as `..`.  
HDI only goes to 2019, so we extrapolate 2020–2023 using each country's recent growth trend.

In [33]:
hdi_raw = pd.read_csv(DATA_RAW / "HDI_1990_2019.csv")
print(f"HDI raw shape: {hdi_raw.shape}")
print(f"Columns: {list(hdi_raw.columns[:5])} ... {list(hdi_raw.columns[-3:])}")
hdi_raw.head(3)

HDI raw shape: (189, 32)
Columns: ['HDI Rank', 'Country', '1990', '1991', '1992'] ... ['2017', '2018', '2019']


,HDI Rank,Country,1990,1991,1992,1993,1994,1995,1996,1997,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,169,Afghanistan,0.302,0.307,0.316,0.312,0.307,0.331,0.335,0.339,...,0.472,0.477,0.489,0.496,0.5,0.5,0.502,0.506,0.509,0.511
1,69,Albania,0.65,0.631,0.615,0.618,0.624,0.637,0.646,0.645,...,0.745,0.764,0.775,0.782,0.787,0.788,0.788,0.790,0.792,0.795
2,91,Algeria,0.572,0.576,0.582,0.586,0.59,0.595,0.602,0.611,...,0.721,0.728,0.728,0.729,0.736,0.74,0.743,0.745,0.746,0.748


In [34]:
# Drop HDI Rank, rename Country
hdi = hdi_raw.drop(columns=["HDI Rank"]).rename(columns={"Country": "country_name"})

# Melt to long format (year columns are 1990..2019)
year_cols = [c for c in hdi.columns if c != "country_name"]
hdi = hdi.melt(id_vars=["country_name"], var_name="year", value_name="hdi")

# Replace ".." with NaN, convert types
hdi["hdi"] = hdi["hdi"].replace("..", np.nan).astype(float)
hdi["year"] = hdi["year"].astype(int)

# Map country names to codes
hdi["country_code"] = hdi["country_name"].map(name_to_code)

unmatched_hdi = hdi[hdi["country_code"].isna()]["country_name"].unique()
if len(unmatched_hdi) > 0:
    print(f"HDI countries not matched to WB codes: {len(unmatched_hdi)}")
    print(f"Examples: {unmatched_hdi[:10]}")

# Filter to 1996-2019 (we'll extrapolate 2020-2023 next)
hdi = hdi.dropna(subset=["country_code"])
hdi = hdi[hdi["year"] >= 1996].reset_index(drop=True)

print(f"HDI shape: {hdi.shape}, Countries: {hdi['country_code'].nunique()}")

HDI countries not matched to WB codes: 24
Examples: ['Bahamas' 'Bolivia (Plurinational State of)' 'Congo'
 'Congo (Democratic Republic of the)' "Côte d'Ivoire" 'Egypt'
 'Eswatini (Kingdom of)' 'Gambia' 'Hong Kong, China (SAR)'
 'Iran (Islamic Republic of)']
HDI shape: (3960, 4), Countries: 165


### Extrapolate HDI for 2020–2023

For each country, compute the geometric mean of HDI growth rates over the last 5 years (2015–2019),  
then project forward year by year.

In [35]:
def extrapolate_hdi(group):
    """Extrapolate HDI from 2019 to 2023 using geometric mean of recent growth."""
    code = group["country_code"].iloc[0]
    name = group["country_name"].iloc[0]
    
    # Get 2015-2019 values
    recent = group[group["year"].between(2015, 2019)].sort_values("year")
    recent_vals = recent["hdi"].dropna()
    
    if len(recent_vals) < 2:
        # Not enough data to compute growth — leave 2020-2023 as NaN
        new_rows = pd.DataFrame({
            "country_name": name, "country_code": code,
            "year": [2020, 2021, 2022, 2023], "hdi": np.nan
        })
        return pd.concat([group, new_rows], ignore_index=True)
    
    # Year-over-year growth rates
    vals = recent_vals.values
    growth_rates = vals[1:] / vals[:-1]
    
    # Geometric mean of growth rates
    avg_growth = np.exp(np.mean(np.log(growth_rates)))
    
    # Cap growth to avoid runaway values
    avg_growth = np.clip(avg_growth, 0.99, 1.05)
    
    # Get last known value
    last_val = group.loc[group["year"] == 2019, "hdi"].values
    if len(last_val) == 0 or np.isnan(last_val[0]):
        last_val = recent_vals.iloc[-1]
    else:
        last_val = last_val[0]
    
    # Project forward
    new_rows = []
    for i, yr in enumerate([2020, 2021, 2022, 2023], start=1):
        projected = last_val * (avg_growth ** i)
        projected = min(projected, 1.0)  # HDI is capped at 1.0
        new_rows.append({
            "country_name": name, "country_code": code,
            "year": yr, "hdi": projected
        })
    
    return pd.concat([group, pd.DataFrame(new_rows)], ignore_index=True)

hdi = hdi.groupby("country_code", group_keys=False).apply(extrapolate_hdi).reset_index(drop=True)
print(f"HDI after extrapolation: {hdi.shape}")
print(f"Year range: {hdi['year'].min()}–{hdi['year'].max()}")
print(f"HDI non-null 2020-2023: {hdi[hdi['year'] >= 2020]['hdi'].notna().sum()}")

HDI after extrapolation: (4620, 4)
Year range: 1996–2023
HDI non-null 2020-2023: 660


## Load Population from SSP Excel

The GDP file gives total GDP (current US$). To compute GDP per capita, we need population.  
We pull population from the SSP Excel file's **Historical Reference** scenario.  

The Excel data is in 5-year intervals (1950, 1955, ..., 2100), so we interpolate to annual.

In [36]:
ssp_raw = pd.read_excel(
    DATA_RAW / "GDP(Forecast)_POP_SSP_1950_2100.xlsx",
    sheet_name="data"
)
print(f"SSP raw shape: {ssp_raw.shape}")
print(f"Columns: {list(ssp_raw.columns[:8])}")
print(f"Variables: {ssp_raw['Variable'].unique()[:5]}")
print(f"Scenarios: {ssp_raw['Scenario'].unique()}")
ssp_raw.head(3)

SSP raw shape: (343259, 36)
Columns: ['Model', 'Scenario', 'Region', 'Variable', 'Unit', '1950', '1955', '1960']
Variables: ['GDP|PPP' 'Population' 'Population|Female' 'Population|Female|Age 0-4'
 'Population|Female|Age 10-14']
Scenarios: ['SSP1' 'SSP2' 'SSP3' 'SSP4' 'SSP5' 'Historical Reference']


,Model,Scenario,Region,Variable,Unit,1950,1955,1960,1965,1970,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
0,IIASA GDP 2023,SSP1,Afghanistan,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,327.105955,423.640014,527.886110,637.688750,745.606295,846.558705,937.135061,1018.194282,1090.567501,1127.372365
1,IIASA GDP 2023,SSP1,Africa (R10),GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,37090.900810,45893.102374,55206.857010,64686.171217,74002.250157,82862.017818,91101.053099,98692.515711,105629.637455,109475.736120
2,IIASA GDP 2023,SSP1,Albania,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,79.001085,83.924946,90.601621,96.085551,100.143830,102.783202,104.634990,105.650176,106.269104,104.419453


In [37]:
# Filter: Historical Reference, total Population
pop_ssp = ssp_raw[
    (ssp_raw["Scenario"] == "Historical Reference") &
    (ssp_raw["Variable"] == "Population")
].copy()

print(f"Population unit: {pop_ssp['Unit'].iloc[0]}")
print(f"Population rows: {len(pop_ssp)}")

# Year columns can be str ('1950'), int (1950), or float (1950.0) depending on reader
def _is_year_col(c):
    try:
        return 1900 <= int(float(str(c))) <= 2200
    except (ValueError, TypeError):
        return False

year_cols_ssp = [c for c in pop_ssp.columns if _is_year_col(c)]
print(f"Year columns found: {len(year_cols_ssp)}")
print(f"First 5: {year_cols_ssp[:5]}")
print(f"Column types: {[type(c).__name__ for c in year_cols_ssp[:3]]}")

pop_long = pop_ssp[["Region"] + year_cols_ssp].melt(
    id_vars=["Region"], var_name="year", value_name="population"
)
print(f"After melt: {pop_long.shape}, non-null pop: {pop_long['population'].notna().sum()}")

pop_long["year"] = pop_long["year"].astype(float).astype(int)
pop_long["population"] = pd.to_numeric(pop_long["population"], errors="coerce")

print(f"Pop long shape: {pop_long.shape}")
pop_long.head()

Population unit: million
Population rows: 225
Year columns found: 31
First 5: ['1950', '1955', '1960', '1965', '1970']
Column types: ['str', 'str', 'str']
After melt: (6975, 3), non-null pop: 3375
Pop long shape: (6975, 3)


,Region,year,population
0,Afghanistan,1950,7.436561
1,Africa (R10),1950,225.115119
2,Albania,1950,1.234474
3,Algeria,1950,8.892792
4,Angola,1950,4.432912


### Map SSP country names to World Bank codes and interpolate to annual

In [38]:
# The SSP file uses country names in the "Region" column.
# We map these to WB country codes using the GDP-derived mapping.
# Some names differ between SSP and WB — we handle common mismatches.

SSP_NAME_FIXES = {
    "Bolivia (Plurinational State of)": "Bolivia",
    "Bonaire, Sint Eustatius and Saba": "Bonaire, Sint Eustatius and Saba",
    "Brunei Darussalam": "Brunei Darussalam",
    "Cabo Verde": "Cabo Verde",
    "China, Hong Kong SAR": "Hong Kong SAR, China",
    "China, Macao SAR": "Macao SAR, China",
    "China, Taiwan Province of China": "Taiwan, China",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Curaçao": "Curacao",
    "Czechia": "Czech Republic",
    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Eswatini": "Eswatini",
    "Gambia": "Gambia, The",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Lao People's Democratic Republic": "Lao PDR",
    "Micronesia (Federated States of)": "Micronesia, Fed. Sts.",
    "Moldova (Republic of)": "Moldova",
    "North Macedonia": "North Macedonia",
    "Republic of Korea": "Korea, Rep.",
    "Republic of the Congo": "Congo, Rep.",
    "Russian Federation": "Russian Federation",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Slovakia": "Slovak Republic",
    "State of Palestine": "West Bank and Gaza",
    "Syrian Arab Republic": "Syrian Arab Republic",
    "Tanzania (United Republic of)": "Tanzania",
    "Timor-Leste": "Timor-Leste",
    "Türkiye": "Turkiye",
    "United Kingdom": "United Kingdom",
    "United States of America": "United States",
    "Venezuela (Bolivarian Republic of)": "Venezuela, RB",
    "Viet Nam": "Vietnam",
    "Yemen": "Yemen, Rep.",
}

# Apply name fixes
pop_long["country_name_wb"] = pop_long["Region"].map(
    lambda x: SSP_NAME_FIXES.get(x, x)
)
pop_long["country_code"] = pop_long["country_name_wb"].map(name_to_code)

# Report unmatched
unmatched_pop = pop_long[pop_long["country_code"].isna()]["Region"].unique()
print(f"SSP regions not matched: {len(unmatched_pop)}")
if len(unmatched_pop) > 0:
    print(f"Examples: {list(unmatched_pop[:15])}")

pop_long = pop_long.dropna(subset=["country_code"])
print(f"Matched population entries: {len(pop_long)}")

SSP regions not matched: 50
Examples: ['Africa (R10)', 'Asia (R5)', 'Bahamas', 'China (R9)', 'China+ (R10)', 'Congo', 'Czechia', 'Europe (R10)', 'European Union (R9)', 'French Guiana', 'Guadeloupe', 'Hong Kong', 'India (R9)', 'India+ (R10)', 'Iran']
Matched population entries: 5425


In [39]:
# Interpolate from 5-year intervals to annual
# Keep data from 1990 to 2025 (broader than 1996-2023 for smooth interpolation)
pop_5yr = pop_long[
    (pop_long["year"] >= 1990) & (pop_long["year"] <= 2025)
][["country_code", "year", "population"]].copy()

def interpolate_to_annual(group):
    """Interpolate 5-year population data to annual."""
    code = group["country_code"].iloc[0]
    group = group.set_index("year").sort_index()
    
    # Create annual index
    full_years = pd.RangeIndex(start=group.index.min(), stop=group.index.max() + 1)
    group = group.reindex(full_years)
    group["population"] = group["population"].interpolate(method="linear")
    group["country_code"] = code
    group.index.name = "year"
    return group.reset_index()

pop_annual = (
    pop_5yr.groupby("country_code", group_keys=False)
    .apply(interpolate_to_annual)
    .reset_index(drop=True)
)

# Filter to 1996-2023
pop_annual = pop_annual[
    (pop_annual["year"] >= 1996) & (pop_annual["year"] <= 2023)
].reset_index(drop=True)

print(f"Annual population: {pop_annual.shape}, Countries: {pop_annual['country_code'].nunique()}")
pop_annual.head()

Annual population: (4900, 3), Countries: 175


,year,country_code,population
0,1996,ABW,0.078332
1,1997,ABW,0.080779
2,1998,ABW,0.083227
3,1999,ABW,0.085674
4,2000,ABW,0.088122


## Merge all variables into one panel

Join everything on `(country_code, year)`. Start with GDP (broadest coverage), then left-join each indicator.

In [40]:
# Start with the full country-year grid from GDP
panel = gdp[["country_name", "country_code", "year"]].copy()
panel = panel.merge(gdp[["country_code", "year", "gdp"]], on=["country_code", "year"], how="left")

# Merge population
panel = panel.merge(pop_annual[["country_code", "year", "population"]], on=["country_code", "year"], how="left")

# Merge each indicator
for df, cols in [
    (corruption, ["country_code", "year", "control_of_corruption"]),
    (agriculture, ["country_code", "year", "employment_agriculture"]),
    (gini, ["country_code", "year", "gini"]),
    (hdi, ["country_code", "year", "hdi"]),
    (pov3, ["country_code", "year", "poverty_3"]),
    # (pov420, ["country_code", "year", "poverty_4_20"]),  # EXCLUDED: poverty gap, not headcount ratio
    (pov830, ["country_code", "year", "poverty_8_30"]),
    (pov10, ["country_code", "year", "poverty_10"]),
]:
    panel = panel.merge(df[cols], on=["country_code", "year"], how="left")

print(f"Panel shape: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}, Years: {panel['year'].min()}-{panel['year'].max()}")
panel.head()

Panel shape: (6076, 12)
Countries: 217, Years: 1996-2023


,country_name,country_code,year,gdp,population,control_of_corruption,employment_agriculture,gini,hdi,poverty_3,poverty_8_30,poverty_10
0,Aruba,ABW,1996,1.379888e+09,0.078332,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1996,NaN,16.791414,-1.291705,67.170502,NaN,0.335,NaN,NaN,NaN
2,Angola,AGO,1996,7.526422e+09,14.170120,-1.167702,38.959080,NaN,NaN,NaN,NaN,NaN
3,Albania,ALB,1996,3.234486e+09,3.271111,-0.893903,52.857857,27.0,0.646,3.0,58.611904,71.59013
4,Andorra,AND,1996,1.224024e+09,NaN,1.318143,NaN,NaN,NaN,NaN,NaN,NaN


### Compute GDP per capita

In [41]:
# Check population unit from SSP file
# SSP population is in 'million' — so multiply by 1e6 before dividing
print(f"GDP example (USA 2020): {panel.loc[(panel['country_code']=='USA') & (panel['year']==2020), 'gdp'].values}")
print(f"Pop example (USA 2020): {panel.loc[(panel['country_code']=='USA') & (panel['year']==2020), 'population'].values}")

# Population is in millions, GDP is in current USD
panel["gdp_per_capita"] = panel["gdp"] / (panel["population"] * 1e6)

# Sanity check
print(f"\nGDP per capita (USA 2020): ${panel.loc[(panel['country_code']=='USA') & (panel['year']==2020), 'gdp_per_capita'].values}")
print("(Should be ~63,000 for USA in 2020)")
print(f"\nGDP per capita non-null: {panel['gdp_per_capita'].notna().sum()}")

GDP example (USA 2020): [2.10604736e+13]
Pop example (USA 2020): [335.388236]

GDP per capita (USA 2020): $[62794.31223998]
(Should be ~63,000 for USA in 2020)

GDP per capita non-null: 4844


> **Note:** If the GDP per capita sanity check above shows unexpected values, adjust the population
> scaling factor. SSP population units vary — check the `Unit` column printed earlier.
> - If unit is `million`: use `population * 1e6`
> - If unit is `thousand`: use `population * 1e3`

## Record NaN fractions before imputation

We track how much of each row was missing *before* imputation, so we can discuss data quality.

In [42]:
# Feature columns (what we'll use in modelling later)
feature_cols = ["gdp_per_capita", "hdi", "control_of_corruption", "employment_agriculture", "gini"]
target_cols = ["poverty_3", "poverty_8_30", "poverty_10"]  # poverty_4_20 excluded (poverty gap data)
all_indicator_cols = feature_cols + target_cols

# NaN summary BEFORE imputation
nan_before = panel[all_indicator_cols].isna().mean()
print("=== % NaN BEFORE imputation ===")
print(nan_before.round(4) * 100)

# Per-row imputation fraction
panel["imputation_pct"] = panel[all_indicator_cols].isna().mean(axis=1)

=== % NaN BEFORE imputation ===
gdp_per_capita            20.28
hdi                       27.57
control_of_corruption     17.91
employment_agriculture    13.89
gini                      67.10
poverty_3                 67.10
poverty_8_30              72.42
poverty_10                72.42
dtype: float64


<cell_type>markdown</cell_type>## Imputation (features only)

**Target columns (poverty) are NOT imputed** — the model should train on real observations only.

Strategy for features (per country, per variable):
1. **Linear interpolation** between existing values
2. **Forward-fill** (max 2 years) and **backward-fill** (max 2 years)
3. **Year-median** for remaining NaN (preserves temporal structure)
4. **Nearest-year median** as final fallback (interpolated from adjacent years)

We do NOT drop any countries — countries with poor data coverage should be kept  
to avoid biasing the dataset against poor nations.

In [43]:
# Sort for proper interpolation
panel = panel.sort_values(["country_code", "year"]).reset_index(drop=True)

# Only impute FEATURES, not targets — targets should retain real observed values only
# so that the ML model trains on actual poverty data, not synthetic medians.
for col in feature_cols:
    panel[col] = (
        panel.groupby("country_code")[col]
        .transform(lambda x: x.interpolate(method="linear", limit_area="inside"))
    )
    panel[col] = (
        panel.groupby("country_code")[col]
        .transform(lambda x: x.ffill(limit=2))
    )
    panel[col] = (
        panel.groupby("country_code")[col]
        .transform(lambda x: x.bfill(limit=2))
    )

print("After interpolation + ffill/bfill (features only):")
print((panel[all_indicator_cols].isna().mean() * 100).round(2))

After interpolation + ffill/bfill (features only):
gdp_per_capita            19.98
hdi                       26.42
control_of_corruption      8.08
employment_agriculture    13.82
gini                      36.93
poverty_3                 67.10
poverty_8_30              72.42
poverty_10                72.42
dtype: float64


In [44]:
# Step 3: Fill remaining NaN with year-median (FEATURES ONLY)
# This preserves temporal structure — a missing value in 1996 gets the 1996 median,
# not a global median that blends 1996–2023 together.
for col in feature_cols:
    year_medians = panel.groupby("year")[col].transform("median")
    panel[col] = panel[col].fillna(year_medians)

# Final fallback: nearest-year median (for any year that has no data at all)
for col in feature_cols:
    if panel[col].isna().any():
        # Build year→median lookup, forward/back fill gaps between years
        yr_med = panel.groupby("year")[col].median()
        yr_med = yr_med.interpolate(method="linear").ffill().bfill()
        panel[col] = panel[col].fillna(panel["year"].map(yr_med))

print("After year-median fill (features only):")
print((panel[all_indicator_cols].isna().mean() * 100).round(2))

After year-median fill (features only):
gdp_per_capita             0.00
hdi                        0.00
control_of_corruption      0.00
employment_agriculture     0.00
gini                       0.00
poverty_3                 67.10
poverty_8_30              72.42
poverty_10                72.42
dtype: float64


## Save output

In [45]:
# Final column selection and order
output_cols = [
    "country_name", "country_code", "year",
    "gdp_per_capita", "hdi", "control_of_corruption",
    "employment_agriculture", "gini",
    "poverty_3", "poverty_8_30", "poverty_10",  # poverty_4_20 excluded
    "imputation_pct"
]
panel = panel[output_cols]

panel.to_csv(DATA_PROCESSED / "historical_panel.csv", index=False)
print(f"Saved: {DATA_PROCESSED / 'historical_panel.csv'}")
print(f"Shape: {panel.shape}")

Saved: ../data/processed/historical_panel.csv
Shape: (6076, 12)


## Summary

In [46]:
print("=" * 60)
print("HISTORICAL PANEL SUMMARY")
print("=" * 60)
print(f"Countries:  {panel['country_code'].nunique()}")
print(f"Year range: {panel['year'].min()} – {panel['year'].max()}")
print(f"Total rows: {len(panel)}")
print()
print("% NaN BEFORE imputation:")
print((nan_before * 100).round(2))
print()
print("% NaN AFTER imputation:")
print((panel[all_indicator_cols].isna().mean() * 100).round(2))
print()
print("Average imputation_pct per row:", panel["imputation_pct"].mean().round(4))
print()
print(panel.describe().round(3))

HISTORICAL PANEL SUMMARY
Countries:  217
Year range: 1996 – 2023
Total rows: 6076

% NaN BEFORE imputation:
gdp_per_capita            20.28
hdi                       27.57
control_of_corruption     17.91
employment_agriculture    13.89
gini                      67.10
poverty_3                 67.10
poverty_8_30              72.42
poverty_10                72.42
dtype: float64

% NaN AFTER imputation:
gdp_per_capita             0.00
hdi                        0.00
control_of_corruption      0.00
employment_agriculture     0.00
gini                       0.00
poverty_3                 67.10
poverty_8_30              72.42
poverty_10                72.42
dtype: float64

Average imputation_pct per row: 0.4483

           year  gdp_per_capita       hdi  control_of_corruption  \
count  6076.000        6076.000  6076.000               6076.000   
mean   2009.500       10345.670     0.693                 -0.039   
std       8.078       16360.465     0.142                  0.959   
min    1996.

---
**Output produced:** `data/processed/historical_panel.csv`  
Contains one row per country-year (1996–2023) with 5 features, 3 poverty targets, and imputation quality flag.

> `poverty_4_20` excluded — source file contains poverty gap, not headcount ratio.